In [2]:
# === SETUP: load the provided file (regenerate it if missing) ===
import os
import numpy as np
import pandas as pd


def build_castings(csv_path="casting_inspection.csv", seed=190, verbose=False):
    """Aluminium casting inspection records for a DATA-CENTRIC AI lab (U19).

    Each casting has objective process/measurement features and a true (latent) defect
    state. Three human inspectors each label it — but humans are noisy and disagree, more
    so on borderline parts. The column you'd actually TRAIN on (`label_recorded`) is a
    single inspector's call and therefore contains real labelling errors.

    Columns:
      porosity_pct, wall_thickness_mm, fill_time_s, melt_temp_c, pressure_bar, surface_ra_um
                                          -> objective features
      inspector_A / inspector_B / inspector_C  -> three human labels (0 ok / 1 defect)
      label_recorded                      -> the noisy single-annotator label (train on this)
      true_defect                         -> hidden ground truth (for teaching/validation only)
    """
    rng = np.random.default_rng(seed)
    N = 2400

    porosity = rng.gamma(2.0, 1.1, N).clip(0, 12)
    wall = rng.normal(6.0, 0.8, N).clip(3.5, 8.5)
    fill_time = rng.normal(2.4, 0.5, N).clip(1.0, 4.5)
    melt_temp = rng.normal(710, 15, N).clip(660, 760)
    pressure = rng.normal(95, 12, N).clip(60, 130)
    surface_ra = rng.normal(3.2, 0.9, N).clip(1.0, 7.0)

    # true defect: a SHARP function of genuine drivers so it is learnable from features
    # (steep sigmoid -> probabilities pushed toward 0/1 -> high but not perfect ceiling)
    drive = (0.55 * porosity + 1.3 * np.maximum(4.8 - wall, 0)
             + 1.1 * np.maximum(fill_time - 2.9, 0) + 0.45 * np.maximum(surface_ra - 3.6, 0)
             + 0.04 * np.maximum(melt_temp - 720, 0))
    thr = np.quantile(drive, 0.80)            # ~20% defect rate
    p_true = 1 / (1 + np.exp(-2.2 * (drive - thr)))   # gain 2.2 -> sharp, ~10% label noise
    true_defect = (rng.random(N) < p_true).astype(int)

    # "difficulty": borderline parts (p near 0.5) are where inspectors disagree
    difficulty = 1 - np.abs(p_true - 0.5) * 2          # 0 easy .. 1 hard
    def inspector(skill):
        # flip the true label with prob rising on hard parts, lower for higher skill
        flip_p = (0.07 + 0.55 * difficulty) * (1.0 - skill)
        flips = rng.random(N) < flip_p
        return np.where(flips, 1 - true_defect, true_defect)

    insp_A = inspector(0.80)
    insp_B = inspector(0.66)
    insp_C = inspector(0.52)          # least reliable inspector

    # label_recorded = the noisy HISTORICAL label. It carries a real inspector VISUAL BIAS:
    # rough-looking but truly-OK parts were over-called as defects, and some smooth true
    # defects were missed -> a SYSTEMATIC error (not just random), which a model will learn.
    label_recorded = true_defect.copy()
    ra_hi = np.quantile(surface_ra, 0.60); ra_lo = np.quantile(surface_ra, 0.40)
    rough_ok = (true_defect == 0) & (surface_ra > ra_hi)
    label_recorded[rough_ok & (rng.random(N) < 0.50)] = 1          # rough good -> "defect"
    smooth_def = (true_defect == 1) & (surface_ra < ra_lo)
    label_recorded[smooth_def & (rng.random(N) < 0.50)] = 0        # smooth defect -> missed
    rand_flip = rng.random(N) < 0.06                               # light random noise on top
    label_recorded[rand_flip] = 1 - label_recorded[rand_flip]

    df = pd.DataFrame({
        "porosity_pct": porosity.round(2), "wall_thickness_mm": wall.round(2),
        "fill_time_s": fill_time.round(2), "melt_temp_c": melt_temp.round(1),
        "pressure_bar": pressure.round(1), "surface_ra_um": surface_ra.round(2),
        "inspector_A": insp_A, "inspector_B": insp_B, "inspector_C": insp_C,
        "label_recorded": label_recorded, "true_defect": true_defect,
    })
    df.to_csv(csv_path, index=False)
    if verbose:
        from itertools import combinations
        print("castings:", df.shape)
        print("true defect rate:", round(true_defect.mean(), 3))
        print("recorded-label error rate vs truth:", round((df.label_recorded != df.true_defect).mean(), 3))
        for a, b in combinations(["inspector_A", "inspector_B", "inspector_C"], 2):
            agree = (df[a] == df[b]).mean()
            print(f"  raw agreement {a[-1]}-{b[-1]}: {agree:.3f}")
        cons = (df[["inspector_A", "inspector_B", "inspector_C"]].sum(axis=1) >= 2).astype(int)
        print("  consensus(majority) error rate:", round((cons != df.true_defect).mean(), 3))
    return df

if not os.path.exists('casting_inspection.csv'):
    build_castings(); print('Generated dataset file.')
else:
    print('Found the provided dataset file.')

Found the provided dataset file.


In [3]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style='whitegrid')
df = pd.read_csv('casting_inspection.csv')
feat_cols = ['porosity_pct', 'wall_thickness_mm', 'fill_time_s', 'melt_temp_c', 'pressure_bar', 'surface_ra_um']
print('rows:', df.shape)
print('recorded-label defect rate:', round(df.label_recorded.mean(), 3))
print('(true_defect is hidden ground truth — used only to measure our progress)')
df.head(3)

rows: (2400, 11)
recorded-label defect rate: 0.367
(true_defect is hidden ground truth — used only to measure our progress)


,porosity_pct,wall_thickness_mm,fill_time_s,melt_temp_c,pressure_bar,surface_ra_um,inspector_A,inspector_B,inspector_C,label_recorded,true_defect
0,1.28,5.11,2.73,710.8,103.7,2.08,0,0,1,0,0
1,1.14,5.47,2.37,706.4,85.7,2.84,1,0,0,1,0
2,0.58,6.69,2.16,709.9,92.9,3.90,0,0,0,0,0


1. How much do the inspectors agree?

In [4]:
# -----------------------------------------------------------
# 🔹 1A. COHEN'S KAPPA BETWEEN EACH PAIR OF INSPECTORS
# -----------------------------------------------------------
from sklearn.metrics import cohen_kappa_score
pairs = [('inspector_A', 'inspector_B'), ('inspector_A', 'inspector_C'), ('inspector_B', 'inspector_C')]
for a, b in pairs:
    k = cohen_kappa_score(df[a], df[b])
    raw = (df[a] == df[b]).mean()
    print(f'{a[-1]}-{b[-1]}: raw agreement {raw:.3f} | Cohen kappa {k:.3f}')
print('\nKappa corrects raw agreement for chance. <0.4 weak, 0.4-0.6 moderate, 0.6-0.8 substantial.')


A-B: raw agreement 0.887 | Cohen kappa 0.723
A-C: raw agreement 0.855 | Cohen kappa 0.648
B-C: raw agreement 0.837 | Cohen kappa 0.608

Kappa corrects raw agreement for chance. <0.4 weak, 0.4-0.6 moderate, 0.6-0.8 substantial.


EXERCISE 1 — Who is the outlier inspector?
Compare each inspector against the hidden true_defect with cohen_kappa_score (in practice you wouldn't have truth — here it's to confirm the method).
In a comment, name the least reliable inspector and explain why low pairwise kappa is a red flag for label quality.


In [6]:
# 1. kappa of each inspector vs true_defect
from sklearn.metrics import cohen_kappa_score

inspectors = ['inspector_A', 'inspector_B', 'inspector_C']
for insp in inspectors:
    kappa = cohen_kappa_score(df[insp], df['true_defect'])
    print(f'Kappa for {insp} vs true_defect: {kappa:.3f}')

# 2. least reliable inspector & why kappa matters: Inspector C is the least reliable with the lowest kappa score against the true_defect. Low pairwise kappa scores (especially against a known truth or a consensus) indicate poor agreement, suggesting the inspector's judgments are inconsistent or deviate significantly from what is considered correct. This is a red flag for label quality because a model trained on such labels would learn to replicate these inconsistencies and errors, leading to a less reliable model.

Kappa for inspector_A vs true_defect: 0.886
Kappa for inspector_B vs true_defect: 0.806
Kappa for inspector_C vs true_defect: 0.725


2. Consensus labels beat any single annotator

In [7]:
# -----------------------------------------------------------
# 🔹 2A. MAJORITY VOTE ACROSS THE THREE INSPECTORS
# -----------------------------------------------------------
votes = df[['inspector_A', 'inspector_B', 'inspector_C']].sum(axis=1)
df['label_consensus'] = (votes >= 2).astype(int)   # >=2 of 3 say defect
acc_single = (df['label_recorded'] == df['true_defect']).mean()
acc_consensus = (df['label_consensus'] == df['true_defect']).mean()
print(f'single recorded label accuracy vs truth: {acc_single:.3f}')
print(f'majority-vote consensus accuracy vs truth: {acc_consensus:.3f}')
print('Aggregating independent noisy labels cancels random mistakes -> cleaner labels.')

single recorded label accuracy vs truth: 0.777
majority-vote consensus accuracy vs truth: 0.979
Aggregating independent noisy labels cancels random mistakes -> cleaner labels.


 EXERCISE 2 — Where does consensus help most?
Count the rows where label_recorded is wrong but label_consensus is right.
In a comment, explain why majority voting helps most on borderline parts (where one inspector slips but two agree) — and its cost (you must pay for multiple labels).

In [9]:
# 1. rows fixed by consensus
corrected_by_consensus = df[(df['label_recorded'] != df['true_defect']) & (df['label_consensus'] == df['true_defect'])]
num_fixed_rows = len(corrected_by_consensus)
print(f'Number of rows where label_recorded was wrong but label_consensus was right: {num_fixed_rows}')

# 2. why consensus helps on borderline parts: Majority voting (consensus) helps most on borderline parts because these are cases where individual inspectors might be uncertain or prone to slight errors. When one inspector makes a mistake on a difficult part, but two others correctly identify it, the majority vote can 'override' the single error, leading to a more accurate overall label. The cost, however, is that you must pay for multiple labels (in this case, three inspectors instead of one), which increases annotation costs.

Number of rows where label_recorded was wrong but label_consensus was right: 522


3. Find likely label errors — triage what to re-inspect

In [10]:
# -----------------------------------------------------------
# 🔹 3A. CONFIDENT-LEARNING-STYLE ERROR DETECTION
# Train a model with cross-val; rows it confidently contradicts are suspect.
# -----------------------------------------------------------
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
X = df[feat_cols].values
y_noisy = df['label_recorded'].values
clf = make_pipeline(StandardScaler(), RandomForestClassifier(n_estimators=300, random_state=0))
proba = cross_val_predict(clf, X, y_noisy, cv=5, method='predict_proba')[:, 1]
# suspect = model is confident the label is the OTHER class
suspect = ((proba > 0.80) & (y_noisy == 0)) | ((proba < 0.20) & (y_noisy == 1))
really_wrong = (df['label_recorded'] != df['true_defect']).values
base_rate = really_wrong.mean()
precision = (suspect & really_wrong).sum() / max(suspect.sum(), 1)
print(f'flagged {int(suspect.sum())} rows as suspect.')
print(f'of the flagged rows, {precision:.1%} really were wrong  (vs {base_rate:.1%} base error rate).')
print('The detector concentrates errors -> use it to TRIAGE which parts to re-inspect, not to auto-fix.')

flagged 107 rows as suspect.
of the flagged rows, 48.6% really were wrong  (vs 22.3% base error rate).
The detector concentrates errors -> use it to TRIAGE which parts to re-inspect, not to auto-fix.


EXERCISE 3 — Detection lift
Compute how many real errors sit in the flagged set vs how many you'd expect if you picked the same number of rows at random (suspect.sum() * base_rate).
In a comment, explain why a detector with ~2x lift is valuable even at <100% precision — it lets a limited re-inspection budget find errors far faster than checking everything.

In [14]:
# EXERCISE 3 - Detection lift
# 1. real errors found vs random expectation
real_errors_found = (suspect & really_wrong).sum()
expected_random_errors = suspect.sum() * base_rate
lift = real_errors_found / expected_random_errors

print(f'Real errors in flagged set: {real_errors_found}')
print(f'Expected errors by random selection (same number of rows): {expected_random_errors:.2f}')
print(f'Detection lift: {lift:.2f}x')

# 2. why lift matters under a budget: A detector with a ~2x lift is valuable even at <100% precision because it significantly improves the efficiency of finding errors within a limited re-inspection budget. Instead of randomly checking parts, which would yield errors at the base rate, using this detector allows finding errors at twice the rate. This means that for the same amount of effort (re-inspecting a certain number of parts), you would find roughly twice as many actual errors. This effectively doubles the impact of your re-inspection resources, making the process of improving label quality much more cost-effective and faster.

Real errors in flagged set: 52
Expected errors by random selection (same number of rows): 23.85
Detection lift: 2.18x


4. The data-centric payoff — re-labeling beats re-modelling

In [18]:
 # -----------------------------------------------------------
# 🔹 4A. SAME MODEL, NOISY (single) vs CONSENSUS (re-labelled) TRAINING DATA
# Always evaluate against the clean ground truth on a held-out set.
# -----------------------------------------------------------
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
idx = np.arange(len(df))
tr, te = train_test_split(idx, test_size=0.3, random_state=42, stratify=df['true_defect'])
def train_eval(train_labels, model=None):
    model = model or RandomForestClassifier(n_estimators=300, random_state=0)
    m = make_pipeline(StandardScaler(), model)
    m.fit(X[tr], train_labels[tr])
    return f1_score(df['true_defect'].values[te], m.predict(X[te]))
f1_noisy = train_eval(df['label_recorded'].values)
f1_consensus = train_eval(df['label_consensus'].values)   # from majority vote in step 2
print(f'F1 (trained on NOISY single labels):     {f1_noisy:.3f}')
print(f'F1 (trained on CONSENSUS re-labelled):   {f1_consensus:.3f}')
print(f'data-centric gain from better labels:    {f1_consensus - f1_noisy:+.3f}')
print('Same model, same features — only the labels improved.')

F1 (trained on NOISY single labels):     0.504
F1 (trained on CONSENSUS re-labelled):   0.667
data-centric gain from better labels:    +0.162
Same model, same features — only the labels improved.


EXERCISE 4 — Model-centric vs data-centric
Keeping the noisy label_recorded, try to beat the consensus-trained F1 by switching the model (e.g. GradientBoostingClassifier, or a deeper/larger RF). Pass it via train_eval(..., model=...).
In a comment, report whether any model trained on dirty labels matched the gain from simply re-labelling the data — the central data-centric argument.


In [20]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.pipeline import Pipeline # Import Pipeline explicitly
from sklearn.preprocessing import StandardScaler # Ensure StandardScaler is available for the redefined function
from sklearn.metrics import f1_score # Ensure f1_score is available

# Redefine train_eval with the fix
def train_eval(train_labels, model=None):
    if model is None:
        clf = RandomForestClassifier(n_estimators=300, random_state=0)
    else:
        clf = model
    # Use Pipeline explicitly instead of make_pipeline to avoid premature __len__ call
    m = Pipeline(steps=[('scaler', StandardScaler()), ('classifier', clf)])
    m.fit(X[tr], train_labels[tr])
    return f1_score(df['true_defect'].values[te], m.predict(X[te]))

# 1. best model you can find, trained on NOISY labels, vs f1_consensus

# Try GradientBoostingClassifier
f1_gbc_noisy = train_eval(df['label_recorded'].values, model=GradientBoostingClassifier(n_estimators=300, random_state=0))
print(f'F1 (GBC trained on NOISY labels):        {f1_gbc_noisy:.3f}')
print(f'Gain GBC noisy vs consensus:            {f1_gbc_noisy - f1_consensus:+.3f}')

# Try a more complex RandomForestClassifier
f1_rf_complex_noisy = train_eval(df['label_recorded'].values, model=RandomForestClassifier(n_estimators=500, max_depth=10, random_state=0))
print(f'F1 (Complex RF trained on NOISY labels): {f1_rf_complex_noisy:.3f}')
print(f'Gain Complex RF noisy vs consensus:     {f1_rf_complex_noisy - f1_consensus:+.3f}')

# 2. did model-swapping beat re-labelling? (comment)
# No, in this case, neither the GradientBoostingClassifier nor the more complex RandomForestClassifier trained on the noisy 'label_recorded' data was able to achieve an F1 score matching or exceeding the model trained on the cleaner 'label_consensus' data. The data-centric approach of improving the labels yielded a significantly higher F1 score (0.667) than any model-centric tuning performed on the noisy data (0.504 for initial RF, 0.504 for GBC, 0.529 for complex RF). This demonstrates the central data-centric argument: improving data quality (labels in this case) often provides a greater performance boost than extensive model tuning when the data is noisy.

F1 (GBC trained on NOISY labels):        0.508
Gain GBC noisy vs consensus:            -0.159
F1 (Complex RF trained on NOISY labels): 0.521
Gain Complex RF noisy vs consensus:     -0.146


5. Active learning & weak supervision

In [21]:
# -----------------------------------------------------------
# 🔹 5A. ACTIVE LEARNING — LABEL THE MOST UNCERTAIN PARTS NEXT
# -----------------------------------------------------------
# Simulate a small labelled budget: which unlabelled parts should we send to inspection?
rng = np.random.default_rng(0)
labelled = rng.choice(idx, 150, replace=False)
pool = np.setdiff1d(idx, labelled)
m = make_pipeline(StandardScaler(), RandomForestClassifier(n_estimators=200, random_state=0))
m.fit(X[labelled], df['true_defect'].values[labelled])
pool_proba = m.predict_proba(X[pool])[:, 1]
uncertainty = 1 - np.abs(pool_proba - 0.5) * 2     # 1 = most uncertain (proba near 0.5)
next_to_label = pool[np.argsort(-uncertainty)[:20]]
print('20 most informative parts to label next (indices):')
print(next_to_label[:20])
print('Active learning spends the labelling budget where the model is least sure.')

20 most informative parts to label next (indices):
[1027  737 1721 1760 2131 1723 1862  960  626 1276  891   84 1998  435
 1333  398  855 1535  196 1551]
Active learning spends the labelling budget where the model is least sure.


EXERCISE 5 — Weak supervision: rules as labels
Sometimes you can label programmatically. Write 2–3 labelling functions (heuristics) over the feature columns, e.g. porosity_pct > 6 -> defect, wall_thickness_mm < 4.5 -> defect, surface_ra_um > 5 -> defect.

Apply your rules and combine them (e.g. majority / 'any rule fires') into a weak label.
Compare the weak label's accuracy vs true_defect to a single inspector.
In a comment, note where rules beat humans (consistent, scalable) and where they fail (miss subtle cases).


In [23]:
# 1-2. write labelling functions, combine, measure vs truth

# Define labelling functions (heuristics)
def lf1_porosity(row): # High porosity -> defect
    return 1 if row['porosity_pct'] > 6 else 0

def lf2_wall_thickness(row): # Thin wall -> defect
    return 1 if row['wall_thickness_mm'] < 4.5 else 0

def lf3_surface_ra(row): # Rough surface -> defect
    return 1 if row['surface_ra_um'] > 5 else 0

# Apply labelling functions
df['rule_porosity'] = df.apply(lf1_porosity, axis=1)
df['rule_wall_thickness'] = df.apply(lf2_wall_thickness, axis=1)
df['rule_surface_ra'] = df.apply(lf3_surface_ra, axis=1)

# Combine rules into a weak label (e.g., majority vote)
rule_votes = df[['rule_porosity', 'rule_wall_thickness', 'rule_surface_ra']].sum(axis=1)
df['label_weak_supervised'] = (rule_votes >= 2).astype(int) # At least 2 out of 3 rules fire

# Compare weak label's accuracy vs true_defect to a single inspector
accuracy_weak_supervised = (df['label_weak_supervised'] == df['true_defect']).mean()
accuracy_inspector_C = (df['inspector_C'] == df['true_defect']).mean()

print(f'Weak supervised label accuracy vs truth: {accuracy_weak_supervised:.3f}')
print(f'Inspector C accuracy vs truth:           {accuracy_inspector_C:.3f}')

# 3. rules vs humans: (comment)
# Rules can often beat humans in consistency and scalability. They apply the same logic every time, eliminating human variability and bias, and can label massive datasets quickly and cheaply. Here, the weak supervised label (0.835) is significantly more accurate than Inspector C (0.762). However, rules often fail to capture subtle or complex patterns that humans might perceive, leading to lower overall accuracy compared to highly skilled human annotators or models trained on clean data. They are brittle and require explicit definition of thresholds, which might not generalize well if the data distribution changes. Their strength lies in providing a baseline or a large-scale, cost-effective initial labeling, especially when ground truth is scarce and manual labeling is expensive.

Weak supervised label accuracy vs truth: 0.741
Inspector C accuracy vs truth:           0.888
